In [28]:
import pandas as pd
import numpy as np
import random
from nltk import word_tokenize, edit_distance
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import CountVectorizer
import pymorphy2
from nltk.metrics.distance import edit_distance


In [29]:
def load_words(file_path):
    words = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                word = parts[1].lower()
                words.append(word)
    return words

words = load_words('file.txt')
word_set = set(words)

def edit_distance(s1, s2):
    if len(s1) < len(s2):
        return edit_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    return previous_row[-1]

from collections import defaultdict

word_length_map = defaultdict(list)
for word in words:
    word_length_map[len(word)].append(word)

In [30]:
def correct_typos(sentence, word_set, word_length_map):
    corrected = []
    for original_word in sentence.split():
        word_lower = original_word.lower()
        if word_lower in word_set:
            corrected.append(original_word)
            continue

        candidates = []
        word_len = len(original_word)
        for length in range(max(1, word_len-2), word_len+3):
            candidates.extend(word_length_map.get(length, []))
            
        if not candidates:
            candidates = words
            
        closest = min(candidates, key=lambda x: edit_distance(word_lower, x))
        corrected.append(closest)
    return ' '.join(corrected)

sentence = "Приветству на семинаре по абаботке текста!"
print(correct_typos(sentence, word_set, word_length_map))

приветстви на семина по бабочке текста


In [52]:
from razdel import tokenize

def process_text(text):
    tokens = [tok.text for tok in tokenize(text)]
    
    stemmer = SnowballStemmer('russian')
    stems = [stemmer.stem(token) for token in tokens]
    
    morph = pymorphy2.MorphAnalyzer()
    lemmas = [morph.parse(token)[0].normal_form for token in tokens]
    
    return tokens, stems, lemmas

text = "Съешь ещё этих мягких французских булок"
tokens, stems, lemmas = process_text(text)

print("Токены:", tokens)
print("Стеммы:", stems)
print("Леммы:", lemmas)

Токены: ['Съешь', 'ещё', 'этих', 'мягких', 'французских', 'булок']
Стеммы: ['съеш', 'ещ', 'эт', 'мягк', 'французск', 'булок']
Леммы: ['съесть', 'ещё', 'этот', 'мягкий', 'французский', 'булка']


In [54]:
def text_to_vector(texts):
    vectorizer = CountVectorizer()
    X = vectorizer.fit_transform(texts)
    return X.toarray(), vectorizer.get_feature_names_out()

texts = ["Считайте слова из файла litw-win.txt и запишите их в список words.",
         "В заданном предложении исправьте все опечатки, заменив слова с опечатками на ближайшие (в смысле расстояния Левенштейна) к ним слова из списка words.",
         "Считайте, что в слове есть опечатка, если данное слово не содержится в списке words."]
vectors, features = text_to_vector(texts)
print("Векторы:")
print(vectors)
print("Фичи:", features)

Векторы:
[[1 1 1 1 0 0 0 0 0 0 0 1 1 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 1 1 0]
 [0 0 0 1 1 1 0 0 0 1 1 0 1 1 0 1 1 0 1 0 1 1 1 1 2 0 0 1 0 1 0 0 0 0 0]
 [0 0 0 1 0 0 1 1 1 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 1 1 0 1 0 1 0 1 0 1]]
Фичи: ['litw' 'txt' 'win' 'words' 'ближайшие' 'все' 'данное' 'если' 'есть'
 'заданном' 'заменив' 'запишите' 'из' 'исправьте' 'их' 'левенштейна' 'на'
 'не' 'ним' 'опечатка' 'опечатками' 'опечатки' 'предложении' 'расстояния'
 'слова' 'слове' 'слово' 'смысле' 'содержится' 'списка' 'списке' 'список'
 'считайте' 'файла' 'что']


In [55]:
df = pd.read_csv('preprocessed_descriptions.csv')
all_text = ' '.join(df['description'].astype(str))
words = list(set(word_tokenize(all_text, language='russian')))
print(f"Уникальных слов: {len(words)}")

FileNotFoundError: [Errno 2] No such file or directory: 'preprocessed_descriptions.csv'